# Analyse-Report
### **ZÜRICH TRAM FLOW: 01/2023 – 12/2025**


**Fokus** Wo · Wann · Warum — Verspätungen im Zürcher Tramnetz  
**Ziel** Strukturelle Muster sichtbar machen · Basis für operative Empfehlungen

**Daten-Quelle**  
* IST-Daten von opentransportdata.swiss 
* GTFS- und Meteo-Datan von data.stadt-zuerich.ch

**Daten-Umfang**  
* Exploration: ~94,4 Mio. Zeilen · 16 Tramlinien · 26 Spalten
* Analyse: ~85,4 Mio. Zeilen · 16 Tramlinien · 42 Spalten

**Herausforderungen**
* Datenmenge (3 Jahre)
* Datenqualität (Betriebsbedingt)

**Überblick** 
* Erkenntnisse 
    * Netzstruktur
    * Geografie
    * Temporalität
    * Meteorologie
    * Ereignisse
    * Infrasturkur
* Empfehlungen

---

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics as an
from zh_tram_flow.visualization.insights import (
    plot_monthly_delay_by_line,
    plot_otp_by_line,
    plot_otp_delta_distribution,
    plot_dwell_analysis,
    plot_district_maps,
    plot_infra_maps,
    plot_delay_delta_timeline,
    plot_arrival_vs_departure_timeline,
)

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("04_insights")

## Netzstruktur

### Drei Einschnitte

**Fahrplanwechsel Dez 2023**   
Grösster Netzumbau der VBZ-Geschichte: 10 von 17 Linien umgebaut, L9/L11/L13 mit neuen Streckenführungen und Haltestellen.    

**Baustellen-Ende Jun 2024**     
L12 Streckensperrung mit Ersatzverkehr (Jan 2023–Jun 2024). Ausfallrate 20× über Netzschnitt während der Bauphase.    

**Fahrplanwechsel Dez 2024**    
Stabiler Übergang, geringer Eingriff (L5: +5 Halte).    

#### Befund

**Kein erkennbarer Effekt auf das Verspätungsverhalten**     
* Alle drei Markierungen hinterlassen keinen erkennbaren Knick im Verspätungsverlauf.    
* Veränderte Linien (L11 +5.3s nach Umbau) bewegen sich identisch zu stabilen Referenzlinien (L15 +5.2s).     
* Die VBZ hat erhebliche Netzeingriffe operativ sauber abgewickelt — das Verspätungsniveau folgt saisonalen Mustern, nicht betrieblichen Ereignissen.

In [ ]:
plot_monthly_delay_by_line(lf_clean)

### On-Time-Performance

OTP-Definitio der VBZ:    
**"Anteil aller Ankünfte innerhalb von ±120 Sekunden (±2 Min.) des Fahrplans."**

**OTP Werte netzweit** 
* Über alle Tram-Linien hinweg wird eine OTP von 87% erreicht.
* Bester Wert liegt bei 93,4% der Linie 6
* Schlechtester Wert liegt bei 81,6% der Linie 11

#### Befund
* Das Netz liegt mit 87% rund 8 Prozentpunkte unter dem VBZ-Ziel von 95%.
* Die Spanne reicht von 81.6% (L11) bis 93.4% (L6) — alle Linien verfehlen das Ziel.

In [ ]:
plot_otp_by_line(lf_clean)

### Akkumulierende Verspätungen

**71.5% aller Halte** — Verspätung wächst Stop zu Stop

* 27.2% bauen Verspätung ab · 1.3% neutral
* 71.3% aller Halte: dwell_time = 0s — kein Puffer zum Nachholen

#### Befund

* Verspätungen entstehen nicht punktuell — sie wachsen systematisch über den gesamten Trip
* Fehlende Pufferzeit ist die Ursache: ohne dwell_time kann das Tram verlorene Zeit nicht aufholen
* Das ist ein Fahrplan-Problem, kein Kapazitätsproblem — die Lösung liegt im Design, nicht im Ausbau

In [ ]:
plot_otp_delta_distribution(lf_clean)

In [ ]:
plot_dwell_analysis(lf_clean)

## Geografie

### Delay-Hotspots

**Periphere Aussenkorridore** — nicht zentrale Knotenpunkte

* Friedhof Enzenbühl: 93.8s · Balgrist: 85.2s · Leutschenbach: 82.7s
* Central: 48.3s (15 Linien) · Paradeplatz: 48.2s (14 Linien) — unter dem Netzschnitt

**Kritische Problemzonen**

* K11: 68.3s · OTP 83% · K12: 66.3s — kein Netzausbau 2023

#### Befund

* Hotspots liegen am Stadtrand — viel befahrene Innenstadtknoten performen besser als periphere Endpunkte
* Paradeplatz und Central (je 14–15 Linien) bestätigen: Frequenz allein ist kein Verspätungstreiber

In [ ]:
an.plot_stop_delay_map(lf_clean)

In [ ]:
an.plot_district_analysis(lf_clean)

### Stadtkreise im Vergleich

**Problemzone 1** — K11 · K12 · K8

* K11: 68.3s · OTP 83% · K12: 66.3s · OTP 85% · K8: 63.7s · OTP 85%
* Alle drei liegen 8–18s über dem Netzschnitt — strukturell schwächste Zone

**Problemzone 2** — K9 · K7 · Aussenbezirk

* K9: 59.7s · K7: 58.7s · Aussenbezirk: 58.4s — OTP jeweils 87%
* Erkennbarer Abstand zu Gruppe 1, aber deutlich über den besten Kreisen

**Beste Kreise** — K5 · K10 · K1

* K5: 49.9s · OTP 89% — bester Kreis netzweit
* K1 (Paradeplatz / Innenstadt): 51.3s trotz höchster Haltestellendichte (18.4 Mio. Beobachtungen)

#### Befund

* Zwei klar abgrenzbare Problemzonen — kein gleichmässiges Gefälle über das Netz
* Zentrale Knotenpunkte (K1: 51.3s) performen besser als periphere Aussenkorridore (K11: 68.3s) — Dichte ist kein Problem, Lage ist ein Problem
* OTP-Spanne zwischen bestem (K5: 89%) und schlechtestem Kreis (K11: 83%): 6 Prozentpunkte

In [ ]:
plot_district_maps(lf_clean)

## Temporalität

### Zeitliche Muster

**Abend-Peak** — 21h · 67.9s

* Höchste Verspätung des Tages — Abreisewellen nach Events und Abendveranstaltungen
* Kein Morgenrush: 7h = 48.9s — unter dem Netzschnitt

**Wochentag**

* Schlechtester Tag: Donnerstag · 60.4s · P95 = 194s
* Bester Tag: Sonntag · 48.4s · Montag · 52.3s — Homeoffice-Effekt sichtbar

> **P95** — 95. Perzentil der Ankunftsverspätung: 95% aller Halte liegen *unter* diesem Wert, nur die schlechtesten 5% darüber. Zeigt Extremverspätungen, nicht den Alltag.

#### Befund

* Abend dominiert — kein klassischer Morgenrush im Zürcher Tramnetz
* Wochenende und Montag profitieren von reduziertem MIV — direkter Zusammenhang mit Kfz-Verkehrsdichte
* P95 am Donnerstag (194s): an schlechten Tagen warten Fahrgäste über 3 Minuten auf das Tram

In [ ]:
an.plot_hour_of_day(lf_clean)
an.plot_day_of_week(lf_clean)

## Meteorologie

### Wetter

**Stärkster Einzelfaktor** — Schnee · +54s · OTP −10.9pp

* Regen: +14s · Starkregen: +22s — deutlich schwächer als Schnee

**Geografische Trennung**

* Schnee → Höhenlagen: K10 · K4 · K12
* Starkregen → Flusstäler: K5 (Escher Wyss / Toni-Areal)

**Linien-Paradox**

* L17: Schnee +7.7s · Regen +41.2s — reagiert umgekehrt zur Netztendenz
* L9: Schnee +75.9s · Regen +10.0s — extremste Schnee-Betroffenheit im Netz

#### Befund

* Schnee ist der einzige Wetterfaktor mit zweistelligem OTP-Einbruch (−10.9pp)
* Wetter trifft nicht alle Linien gleich — geografische Lage der Strecke entscheidet über Exposition
* Regen allein ist kein kritischer Faktor — erst Starkregen in Flusstälern zeigt nennenswerten Effekt

In [ ]:
an.plot_weather_overview(lf_clean)

In [ ]:
# Gleiche Farbskala für beide Karten — nur die 3-4 extremen Haltestellen sättigen auf Rot
# vmax=60: Stops mit Δ>60s = tiefrot, Stops mit Δ~20-30s = hellorange
an.plot_weather_stop_map(lf_clean, flag="has_snow",       vmax=60)
an.plot_weather_stop_map(lf_clean, flag="has_heavy_rain", vmax=60)

### Jahreszeit

**Beste Jahreszeit** — Winter · 51.7s · OTP 88.9%

* Herbst: 61.2s — schlechteste Jahreszeit · Sommer: 56.4s · Frühling: 55.6s
* Weniger MIV im Winter → weniger Kreuzungskonflikte — übertrifft den Schnee-Effekt

#### Befund

* Winter ist trotz Schnee die beste Jahreszeit — reduzierter Kfz-Verkehr überwiegt den Schnee-Malus
* Saisonspanne von 9.5s (Winter → Herbst) ist strukturell stabil über alle drei Messjahre
* Herbst kombiniert volles Verkehrsaufkommen mit wechselhaftem Wetter — belastendste Jahreszeit

## Ereignisse

### Feiertage & Events

**Feiertage** — Bester Tag-Typ · 46.3s · OTP 90.6%

* −9.9s gegenüber einem normalen Werktag — reduzierter MIV als Haupttreiber
* Grosse Events: +10.5s — fast ausschliesslich abends 18–22h
* Tagsüber: Event-Tage ≈ Normaltage — kein messbarer Effekt vor 18h

In [ ]:
an.plot_events_overview(lf_clean)

**Schlechteste Kategorie** — Fachmessen · 66.0s · OTP 84%

* Schlechtester Tag: Berufsmesse Zürich 21.11.2024 · 192.5s · OTP 54.5%
* Taylor Swift: 75.4s — Fachmessen schlagen Popkonzerte deutlich

#### Befund

* Feiertage entlasten das Netz stärker als jeder andere Faktor — 46.3s unterschreitet sogar den besten Kreisschnitt
* Fachmessen sind kritischer als Konzerte — andere Zielgruppe, andere Anreisezeit, weniger ÖPNV-Affinität
* Events wirken fast ausschliesslich abends — tageszeitliche Steuerung ist die wirksamste Gegenmassnahme

In [ ]:
an.plot_daily_delay_timeline(lf_delay, cfg)

### Netzausbau 2023

**Ausgebaut** — K3 · K8

* Sihlcity (K3): 55.7s · Rehalp (K8): 63.7s — moderate bis erhöhte Verspätung

**Nicht ausgebaut** — K11 · K12

* K11: 68.3s · OTP 83% · K12: 66.3s — strukturell schlechteste Kreise im Netz

#### Befund

* Null Überschneidung: Investitionsort ≠ Problemort — K8 (ausgebaut) liegt noch vor K11/K12 (nicht ausgebaut)
* Der Ausbau hat die Problemzonen nicht adressiert — K11 und K12 bleiben strukturell unterversorgt
* Nächste Investitionsrunde sollte K11/K12 priorisieren — grösste Hebelwirkung auf OTP und Delay

In [ ]:
plot_infra_maps(lf_delay, lf_clean)

## Empfehlungen

**Strukturelle Muster** — stabil über 3 Jahre

| Priorität | Empfehlung | Basis |
|:---|:---|:---|
| 🔴 Hoch | Puffer einbauen — 10s dwell_time an Aussenkorridor-Halten | 71.5% akkumulieren · 0s geplante Standzeit |
| 🔴 Hoch | Kapazität K11/K12 erhöhen — L11 kritischste Hauptlinie (68.7s) | OTP 83% · strukturell schwächste Zone |
| 🟡 Mittel | Fachmessen-Disposition — L11 Verstärkerkurse an Berufsmesse-Tagen | Schlechtester Tag: 192.5s · OTP 54.5% |
| 🟡 Mittel | Schnee-Protokoll Höhenlagen K10/K4 — Selnau Extremfall +190.9s | L9/L12 bis +76s bei Schnee |
| 🟢 Tief | Nächster Ausbau → K11/K12 statt gut-performende Kreise | 0 Overlap Investitionsort / Problemort |